# Week 6 Day 7 — Build Your First LLM Application

Until now, we've interacted with LLMs through interfaces like ChatGPT. In this notebook, we're seeing what it looks like when **our own Python application** sends a request to an LLM and receives structured information back.

No frameworks, no agents, no vector databases. Just Python, one API call, and the ideas we've been learning: **prompting, context, temperature, structured output, and hallucinations**.

## What are we building?

A tiny **document parser**.

We'll take a short company policy document, send it to an LLM, and ask it to extract four key facts as **structured data** (JSON). Then we'll check that the answer matches the shape we expected.

When you're done, you'll have seen the full pipeline of every real LLM application:

> **Python code → API request → LLM → structured response → back to Python**

## 1. Setup

Real LLM providers protect their API with a **key**. The safe way for a program to use a key is through an **environment variable** — a named value your operating system hands to your program. In a terminal, you would set it like this (macOS / Linux):

```bash
export OPENAI_API_KEY="your-real-key-here"
```

and the program reads it. You **never** paste a real key into a notebook — notebooks get shared, committed to git, and copied around.

For this class we use a small local LLM server (model: `Qwen3.8-27B`), which doesn't check for a key. So the code below reads the `OPENAI_API_KEY` environment variable and falls back to the placeholder string `"none"` if it isn't set. If you later point the notebook at a real provider, that same line will pick up your real key automatically.

Next we install the two libraries we need. If they're already installed, the cell quietly does nothing:

```python
!pip install -q openai pydantic
```

*(If your environment is managed by your school, check with your instructor before running the cell.)*

## 2. Import libraries

Four imports, each with one job:

| Import | What it's for |
| --- | --- |
| `from openai import OpenAI` | the client — how our code talks to the LLM API |
| `from pydantic import BaseModel` | a simple way to describe the shape of the data we expect back |
| `import json` | built-in — turns the LLM's JSON text into a Python dictionary |
| `import os` | built-in — reads environment variables (where the API key lives) |

The `OpenAI` client is the one object we use to make the API call. We point it at our server with `base_url` and hand it the key from the environment variable.

In [ ]:
import os
import json

from openai import OpenAI
from pydantic import BaseModel

# The client is how our program talks to the LLM API.
#
# base_url: where the server lives. The SDK appends the right
# paths for us (for example "/chat/completions"), so we give
# it the base address, not the full URL.
#
# api_key: read from an environment variable. The class server
# doesn't need a real key, so "none" is a safe fallback.
# (Never type a real key directly into the notebook.)
client = OpenAI(
    base_url="http://100.117.48.99:8888/v1",
    api_key=os.environ.get("OPENAI_API_KEY", "none"),
)

## 3. Create the sample document

Here is the document we'll parse — a short company policy. In a real application, a document might come from a file, a database, or a website. For now, we keep it as a plain Python string:

In [ ]:
document = """Remote Work Policy

Employees can work remotely up to 3 days per week.
Remote work requests should be submitted at least 2 days in advance.
Employees must have manager approval."""

print(document)

## 4. Create the prompt

A prompt is just a message we send to the LLM. Ours has two parts:

- **System prompt** — sets the model's role: *"You are a careful fact-extractor. Answer with valid JSON."*
- **User prompt** — the actual task: the document, plus exactly which fields to extract.

Notice that the user prompt spells out each field **and its type** (text, number, true/false). That is the core trick of **structured output**: tell the model the shape of the answer you want.

In [ ]:
system_prompt = (
    "You are a careful assistant. You extract facts from documents "
    "and always answer with valid JSON."
)

user_prompt = f"""Here is a document:

{document}

Extract these details and return them as a single JSON object with exactly these fields:
- "policy_name": the name of the policy (text)
- "remote_days": how many days per week remote work is allowed (number)
- "notice_period_days": how many days in advance a request must be submitted (number)
- "manager_approval": whether manager approval is required (true or false)

Return only the JSON object. Do not include any other text."""

## 5. Define the structured output with Pydantic

A Pydantic `BaseModel` is a class that describes the **shape** of the data we expect. Each field has a name and a type. Think of it as a checklist the LLM's answer must pass:

| Field | Type | Meaning |
| --- | --- | --- |
| `policy_name` | `str` | the name of the policy, e.g. `"Remote Work Policy"` |
| `remote_days` | `int` | allowed remote days per week, e.g. `3` |
| `notice_period_days` | `int` | days in advance a request must be submitted, e.g. `2` |
| `manager_approval` | `bool` | `True` if a manager must approve the request |

If the LLM's answer matches this checklist, Pydantic turns it into a clean `Policy` object our program can use. If a field is missing or has the wrong type, Pydantic raises an error — which is exactly the kind of problem we *want* to catch in a real application.

> **One honest note.** Some APIs let you hand the schema directly to the model (for example, OpenAI's `response_format`). The class server in this notebook doesn't have that feature enabled, so instead we *describe* the shape in the prompt (section 4) and *check* the answer with Pydantic (this section). Same goal, slightly longer route.

In [ ]:
class Policy(BaseModel):
    policy_name: str          # e.g. "Remote Work Policy"
    remote_days: int          # e.g. 3
    notice_period_days: int   # e.g. 2
    manager_approval: bool    # e.g. True

## 6. Make the LLM API call

This is the heart of the notebook — one method call. Its pieces:

- `model` — which model to use (this class uses `Qwen3.8-27B`).
- `messages` — our prompts, as a list of `{role, content}` pairs: the system prompt first, then the user prompt.
- `temperature` — how "creative" the model is. `0` means *as deterministic as possible*. We want the model to **report** what the document says, not improvise, so we set it to `0`.

The call sends an HTTP request to the server and waits for the model's reply in a `response` object. The model's actual text lives at:

`response.choices[0].message.content`

which just reads as: *first (and only) choice → the assistant's message → its text*.

In [ ]:
response = client.chat.completions.create(
    model="Qwen3.8-27B",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ],
    temperature=0,
)

## 7. Print the result

First, let's look at the **raw text** the model wrote. Then we do the interesting part: turn it into a Python dictionary (`json.loads`) and check it against our `Policy` checklist (`Policy.model_validate`). If the check passes, our program holds a real, typed object — not just a string we'd have to guess about.

*(Don't worry about `True` vs `true` — that's just Python's spelling of the same value JSON writes as `true`.)*

In [ ]:
raw_answer = response.choices[0].message.content

print("Raw text from the model:")
print(raw_answer)

In [ ]:
data = json.loads(raw_answer)          # JSON text  ->  Python dictionary
policy = Policy.model_validate(data)   # check it matches our Policy schema

print("Structured result:")
print(policy)

## 8. What just happened?

Let's zoom out. The entire notebook is one pipeline:

```
Document
   "Remote Work Policy..." — a plain string
      ↓
Prompt
   we wrap the document in instructions:
   "extract these 4 fields as JSON"
      ↓
LLM API call
   Python code sends a request to the LLM (Qwen3.8-27B)
      ↓
Structured response
   the model replies with a JSON object
      ↓
Python application
   Pydantic checks it matches the Policy schema,
   and our code holds a real object
```

This is the same pipeline that powers ChatGPT — the only difference is that **we own it**. And every concept from the series just showed up:

- **Prompting** — the system + user prompts tell the model exactly what to do.
- **Context** — the policy document is the context the model reasons over.
- **Temperature** — `0` keeps the model factual and repeatable.
- **Structured output** — asking for JSON (in the prompt) and checking it with Pydantic gives us data a program can rely on.
- **Hallucinations** — one last catch, in the next section.

## 9. A small experiment

Time to play. Pick one:

1. **Change the document.** Edit the `document` string in section 3 — for example, a policy that allows **2** remote days, requires **5** days' notice, and does *not* need manager approval. Leave everything else alone, then re-run the notebook from that cell. Watch the values change while the structure stays the same.
2. **Add a field.** Ask for one more field in the user prompt — for example, `"approval_process": how requests are approved (text)` — and add the same field to the `Policy` model. Re-run and see the new field appear in the result.

## 10. Important reminder: structure is not truth

Here's the part beginners miss most. **Structured output guarantees the *shape* of the answer — not the *truth* of the answer.**

Pydantic can confirm that `remote_days` is a number. It *cannot* confirm that the number is right. If the document says "3 days" and the model confidently returns `remote_days: 5`, Pydantic is still happy — `5` is a valid number. That's a **hallucination**: the model producing something plausible that isn't actually in the document.

So in a real application you would still:

- **validate** the values (for example, "remote_days should be between 0 and 7"), and
- **evaluate** your pipeline (periodically check that extracted answers actually match the source document).

The model is a very confident reader. Your code is the one who double-checks its homework.

## 11. Try it yourself

A few safe, fun experiments. Make your own copy of this notebook, and re-run cells after each change:

1. **Different document.** Change the policy to "2 days remote, 5 days' notice, no manager approval needed" and re-run everything from the document cell. Do the extracted values follow the new document?
2. **Add a field.** Ask for one more field (and add it to the `Policy` model, too). For example, `"approval_process": how requests are approved (text)`.
3. **Ask for something that isn't in the document.** Add a field like `"office_color": the color of the office (text)` to the prompt and the `Policy` model — the document never mentions the office's color. The model will either confidently *invent* a value or admit the document doesn't say. Either way: no answer can be verified from the document, which is exactly the problem real applications have to design around.

One tip: if an experiment makes the model answer in plain English, the `json.loads` cell will raise an error. That's not a bug in your code — it's the model not following your instructions, and it's a great excuse to reword the prompt or lower the temperature.

## Where this goes next

You just built your first LLM application: a Python program that sends a document to an LLM and receives typed, structured data back. Everything we've learned in this series so far — prompting, context, temperature, structured output, hallucinations — is inside this one tiny pipeline.

Next stop: making it **robust** — better validation, checking answers against the source document, and evaluating your prompts.